In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error
from scipy.interpolate import interp1d
from scipy.interpolate import PchipInterpolator
from scipy.interpolate import Akima1DInterpolator

In [3]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [4]:
df = pd.read_csv('C:/Users/Ayyan Aftab/Desktop/iv_surface_reconstruct/data/featured_dataset.csv')
df['datetime'] = pd.to_datetime(df['datetime'])
print(df.shape)
df.head(5)

(26325, 30)


,datetime,underlying_price,contract,iv,option_type,strike,hour,minute,day_of_week,date,session_progress,days_to_expiry,moneyness,log_moneyness,dist_from_atm,dist_from_atm_pct,is_ce,strike_rank,iv_neighbor_-2,iv_neighbor_-1,iv_neighbor_+1,iv_neighbor_+2,iv_neighbor_mean,wide_iv_neighbor_mean,iv_lag_1,iv_roll_mean_5,iv_roll_std_5,iv_roll_mean_10,mean_ce_iv,mean_pe_iv
0,2026-01-07 09:15:00,26111.65,NIFTY27JAN2625200CE,0.12662,CE,25200,9,15,2,2026-01-07,0.000000,20.2569,0.965086,-0.035538,911.65,0.034914,1,0.0,NaN,NaN,0.12330,0.11741,0.12330,0.11741,NaN,NaN,NaN,NaN,0.103060,0.143129
1,2026-01-07 09:20:00,26141.40,NIFTY27JAN2625200CE,0.08632,CE,25200,9,20,2,2026-01-07,0.013514,20.2535,0.963988,-0.036676,941.40,0.036012,1,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.12662,NaN,NaN,NaN,0.098762,0.147267
2,2026-01-07 09:25:00,26139.35,NIFTY27JAN2625200CE,0.09147,CE,25200,9,25,2,2026-01-07,0.027027,20.2500,0.964064,-0.036598,939.35,0.035936,1,0.0,NaN,NaN,NaN,0.09514,NaN,0.09514,0.08632,0.106470,0.028496,NaN,0.091175,0.148739
3,2026-01-07 09:30:00,26128.95,NIFTY27JAN2625200CE,0.10860,CE,25200,9,30,2,2026-01-07,0.040541,20.2465,0.964447,-0.036200,928.95,0.035553,1,0.0,NaN,NaN,0.10842,0.11150,0.10842,0.11150,0.09147,0.101470,0.021932,0.101470,0.103672,0.142787
4,2026-01-07 09:35:00,26131.90,NIFTY27JAN2625200CE,0.10462,CE,25200,9,35,2,2026-01-07,0.054054,20.2431,0.964339,-0.036313,931.90,0.035661,1,0.0,NaN,NaN,0.10538,0.12459,0.10538,0.12459,0.10860,0.103252,0.018259,0.103252,0.104564,0.147425


In [5]:
df['iv'].isna().sum()

np.int64(5279)

In [7]:
np.random.seed(42)

available_rows = df[df['iv'].notna()].index # get rows where IVs are present
hidden_rows = np.random.choice(available_rows,size=int(0.2*len(available_rows)),replace=False) # selecting rows which we're going to hide

ground_truth = df.loc[hidden_rows,'iv']
df_masked = df.copy() 
df_masked.loc[hidden_rows,'iv'] = np.nan

print("Total rows: ",len(df))
print("Available IV rows: ", len(available_rows))
print("Hidden rows: ", len(hidden_rows))

Total rows:  26325
Available IV rows:  21046
Hidden rows:  4209


In [8]:
def eval_pred(actual, predicted):
    valid = ~np.isnan(predicted)

    mse = mean_squared_error(actual[valid],predicted[valid])

    coverage = valid.mean()

    print(f"MSE: {mse:.6f}")
    print(f"Coverage: {coverage:6f}")

    return mse, coverage

In [9]:
ff_df = df_masked.copy()
ff_df = ff_df.sort_values(['option_type','strike','datetime'])

ff_df['iv_ff'] = ff_df.groupby(['option_type','strike'])['iv'].ffill()

ff_pred = ff_df.loc[hidden_rows, 'iv_ff'].values

ff_mse , ff_coverage = eval_pred(ground_truth.values,ff_pred)

MSE: 0.003488
Coverage: 0.998337


In [10]:
rm_pred = df_masked.loc[hidden_rows,'iv_roll_mean_5'].values

rm_mse, rm_coverage = eval_pred(ground_truth.values,rm_pred)

MSE: 0.005476
Coverage: 0.990734


In [11]:
nm_pred = df_masked.loc[hidden_rows,'iv_neighbor_mean'].values

nm_mse, nm_coverage = eval_pred(ground_truth.values,nm_pred)

MSE: 0.000341
Coverage: 0.943692


In [21]:
b4_pred = np.nanmean(np.stack([ff_pred,nm_pred],axis=1),axis=1)

b4_mse, b4_coverage = eval_pred(ground_truth.values,b4_pred)

MSE: 0.001260
Coverage: 0.999762


C:\Users\Ayyan Aftab\AppData\Local\Temp\ipykernel_48472\3524952330.py:1: RuntimeWarning: Mean of empty slice
  b4_pred = np.nanmean(np.stack([ff_pred,nm_pred],axis=1),axis=1)


In [13]:
linear_pred = []

for idx in hidden_rows:
    row = df_masked.loc[idx]

    strikes_sametime = df_masked[(df_masked['datetime'] == row['datetime']) & (df_masked['option_type'] == row['option_type']) & (df_masked['iv'].notna())].sort_values('strike')

    if(len(strikes_sametime) >=3):
        interpolator = interp1d(strikes_sametime['strike'],strikes_sametime['iv'],kind='linear',bounds_error=False)
        pred = float(interpolator(row['strike']))

    else: 
        pred = np.nan

    linear_pred.append(pred)

linear_pred = np.array(linear_pred)

linear_mse, linear_coverage = eval_pred(ground_truth.values,linear_pred)

MSE: 0.000029
Coverage: 0.775481


In [14]:
linear_pred = []

for idx in hidden_rows:
    row = df_masked.loc[idx]

    strikes_sametime = df_masked[(df_masked['datetime'] == row['datetime']) & (df_masked['option_type'] == row['option_type']) & (df_masked['iv'].notna())].sort_values('strike')

    if(len(strikes_sametime) >=3):
        interpolator = interp1d(strikes_sametime['strike'],strikes_sametime['iv'],kind='linear',bounds_error=False,fill_value='extrapolate')
        pred = float(interpolator(row['strike']))

    else: 
        pred = np.nan

    linear_pred.append(pred)

linear_pred = np.array(linear_pred)

linear_mse, linear_coverage = eval_pred(ground_truth.values,linear_pred)

MSE: 0.000137
Coverage: 0.997387


In [15]:
spline_pred = []

for idx in hidden_rows:
    row = df_masked.loc[idx]

    strikes_sametime = df_masked[(df_masked['datetime'] == row['datetime']) & (df_masked['option_type'] == row['option_type']) & (df_masked['iv'].notna())].sort_values('strike')

    if(len(strikes_sametime)>=4):
        interpolator = interp1d(strikes_sametime['strike'],strikes_sametime['iv'],kind='cubic',bounds_error=False)
        pred = float(interpolator(row['strike']))

    else:
        pred = np.nan

    spline_pred.append(pred)

spline_pred = np.array(spline_pred)

spline_mse, spline_coverage = eval_pred(ground_truth.values,spline_pred)

MSE: 0.000023
Coverage: 0.772630


In [16]:
quad_pred = []

for idx in hidden_rows:
    row = df_masked.loc[idx]

    strikes_sametime = df_masked[(df_masked['datetime'] == row['datetime']) & (df_masked['option_type'] == row['option_type']) & (df_masked['iv'].notna())].sort_values('strike')

    if(len(strikes_sametime)>=4):
        interpolator = interp1d(strikes_sametime['strike'],strikes_sametime['iv'],kind='quadratic',bounds_error=False)
        pred = float(interpolator(row['strike']))

    else:
        pred = np.nan

    quad_pred.append(pred)

quad_pred = np.array(quad_pred)

quad_mse, quad_coverage = eval_pred(ground_truth.values,quad_pred)

MSE: 0.000020
Coverage: 0.772630


In [17]:
pchip_pred = []

for idx in hidden_rows:

    row = df_masked.loc[idx]

    strikes_sametime = df_masked[
        (df_masked['datetime'] == row['datetime']) &
        (df_masked['option_type'] == row['option_type']) &
        (df_masked['iv'].notna())
    ].sort_values('strike')

    if len(strikes_sametime) >= 3:

        interpolator = PchipInterpolator(
            strikes_sametime['strike'],
            strikes_sametime['iv'],
            extrapolate=False
        )

        pred = float(interpolator(row['strike']))

    else:
        pred = np.nan

    pchip_pred.append(pred)

pchip_pred = np.array(pchip_pred)

pchip_mse, pchip_coverage = eval_pred(ground_truth.values, pchip_pred)

MSE: 0.000024
Coverage: 0.775481


In [18]:
akima_pred = []

for idx in hidden_rows:

    row = df_masked.loc[idx]

    strikes_sametime = df_masked[
        (df_masked['datetime'] == row['datetime']) &
        (df_masked['option_type'] == row['option_type']) &
        (df_masked['iv'].notna())
    ].sort_values('strike')

    if len(strikes_sametime) >= 5:

        interpolator = Akima1DInterpolator(
            strikes_sametime['strike'],
            strikes_sametime['iv'],
            extrapolate=False
        )

        pred = interpolator(row['strike'])

        pred = float(pred) if pred is not None else np.nan

    else:
        pred = np.nan

    akima_pred.append(pred)

akima_pred = np.array(akima_pred)

akima_mse, akima_coverage = eval_pred(ground_truth.values,akima_pred)

MSE: 0.000019
Coverage: 0.761226
